In [9]:
import pandas as pd
import os

# Replace 'ml-1' with the exact name of your repository folder shown in the left panel
repo_name = "ml-1"
file_path = f"/content/{repo_name}/data/raw/content_refresh_anonymized.csv"

print("Checking file path:", file_path)

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("✅ Data loaded successfully!")
    print(f"Total rows: {len(df)}")
    print(f"Columns available: {df.columns.tolist()}")

Checking file path: /content/ml-1/data/raw/content_refresh_anonymized.csv
✅ Data loaded successfully!
Total rows: 30000
Columns available: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedtarek-5/ml-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents a single, unique content page (`content_id`) evaluated over a specific mid-panel month snapshot (e.g., March 2026). We deliberately avoid using the final month's data to prevent "peeking" into future outcomes and to maintain a strict past-to-future evaluation window.

In [10]:
print("--- Verification: Unit of Analysis & Time Window ---")
print(f"1. Total rows in dataset: {len(df)}")
print(f"2. Unique content pages (Grain): {df['content_id'].nunique()}")

if df['content_id'].is_unique:
    print("VERIFIED: Each row represents exactly one unique content_id.")
else:
    print("NOTE: There are duplicate content_ids (e.g., same page across different months).")

--- Verification: Unit of Analysis & Time Window ---
1. Total rows in dataset: 30000
2. Unique content pages (Grain): 30000
VERIFIED: Each row represents exactly one unique content_id.


## 2. Fields: feature / label / context / excluded

- **Features (Predictors):** `impressions_90d`, `trend_direction`, `content_age_days`, `avg_position`, `ctr`. *(Reason: All are strictly knowable at the decision moment based on historical data).*
- **Label (Target Proxy):** `is_high_opportunity` (A derived binary rule: 1 if trending down with baseline impressions, else 0).
- **Context (Metadata):** `content_id`. *(Reason: Used for grouping and human readability, not fed directly into the model).*
- **Excluded:** Any page with `impressions_90d` < 10. *(Reason: These are likely draft pages, tests, or indexing errors. Reviewing them wastes human time and adds pure noise to the model).*

In [11]:
# Define categories
features = ['impressions_90d', 'trend_direction', 'content_age_days', 'avg_position', 'ctr']
context = ['content_id']

# Verify these columns actually exist in the dataframe to avoid errors
existing_features = [col for col in features if col in df.columns]
existing_context = [col for col in context if col in df.columns]

print("✅ Verified Feature Columns:", existing_features)
print("✅ Verified Context Columns:", existing_context)

# Display a small sample to prove their existence
display(df[existing_context + existing_features].head(3))


✅ Verified Feature Columns: ['impressions_90d', 'trend_direction', 'content_age_days', 'avg_position', 'ctr']
✅ Verified Context Columns: ['content_id']


,content_id,impressions_90d,trend_direction,content_age_days,avg_position,ctr
0,content_304f48230142,3803,down,187,10.6,0.76
1,content_a1fb4e703a9e,15320,down,445,20.3,0.05
2,content_9aa793d4d895,12581,down,141,36.5,0.09


## 3. Verify it with queries (grain, counts, missing values, windows)

Below are the verification queries proving the data contract claims:
1. **Grain & Counts:** Confirming the number of unique pages.
2. **Missing Values:** Checking data completeness for key features.
3. **Availability Check:** Filtering out excluded data (e.g., `impressions_90d` >= 10) to see how many actionable rows survive.

In [12]:
print("--- 1. Grain & Counts ---")
print(f"Total Rows: {len(df)} | Unique Pages: {df['content_id'].nunique()}")

print("\n--- 2. Missing Values Check (Top features) ---")
cols_to_check = [col for col in ['impressions_90d', 'trend_direction', 'content_age_days'] if col in df.columns]
print(df[cols_to_check].isnull().sum())

print("\n--- 3. Availability Check (Exclusion Filter) ---")
if 'impressions_90d' in df.columns:
    actionable_df = df[df['impressions_90d'] >= 10]
    print(f"Total rows before filter: {len(df)}")
    print(f"Rows surviving 'impressions_90d >= 10' filter: {len(actionable_df)}")
    print(f"Percentage of actionable data: {(len(actionable_df)/len(df))*100:.2f}%")
else:
    print("'impressions_90d' column not found. Skipping availability check.")


--- 1. Grain & Counts ---
Total Rows: 30000 | Unique Pages: 30000

--- 2. Missing Values Check (Top features) ---
impressions_90d     0
trend_direction     0
content_age_days    0
dtype: int64

--- 3. Availability Check (Exclusion Filter) ---
Total rows before filter: 30000
Rows surviving 'impressions_90d >= 10' filter: 26254
Percentage of actionable data: 87.51%


## 4. Data limits

1. **No Ground Truth for Uplift:** The dataset is anonymized and static. We lack actual "post-refresh traffic uplift" metrics, forcing us to rely on a proxy label. The model optimizes for this proxy, which is a strong directional indicator, but not a guaranteed measure of future human success.
2. **Historical Bias:** The data only reflects past search algorithm behavior, which may change unexpectedly.
3. **Unbalanced History:** Some pages may have GSC-only early rows with missing engagement metrics (like CTR), creating sparse data windows that the model must handle carefully.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.